# CS331: Zero-Shot & Open-World Sketch-Based Image Retrieval (ZS-SBIR)
## Complete Multi-GPU Pipeline: Train, In-Domain Test & TU-Berlin Cross-Dataset
---
* **Course:** CS331 - Thị giác máy tính nâng cao | Trường ĐH Công nghệ Thông tin - ĐHQG-HCM (UIT)
* **Architecture:** Asymmetric Hybrid LoRA-CLIP (ViT-B/32) + Spatial Patch-Token Attention Pooling
* **Hardware Accelerator:** 2x NVIDIA Tesla T4 (DataParallel + PyTorch AMP FP16)
* **Benchmark Protocol:**
  1. **In-Domain Zero-Shot:** Sketchy Split 2 (104 Seen Training Classes | 21 Isolated Unseen Test Classes - Zero Leakage)
  2. **Cross-Dataset Open-World:** TU-Berlin Extended (21,940 Photos | 2,240 Freehand Memory Sketches)

In [ ]:
# [STEP 1] GPU & Environment Health Check
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
assert torch.cuda.is_available(), "Please enable GPU acceleration (GPU T4 x2) in Kaggle Notebook Settings!"

In [ ]:
# [STEP 2] Clone Repository & Install Academic Dependencies
!git clone -b feat/dual-expert-consensus https://github.com/Quandinh030106/ZS-SBIR-LoRA-CLIP.git || git clone https://github.com/Quandinh030106/ZS-SBIR-LoRA-CLIP.git
%cd ZS-SBIR-LoRA-CLIP
!pip install --quiet git+https://github.com/openai/CLIP.git ftfy regex tqdm matplotlib opencv-python-headless
import clip
print(f"CLIP installed successfully! Available models: {clip.available_models()[:3]}...")

In [ ]:
# [STEP 3] Run Comprehensive Pipeline Unit Self-Test (Zero Failure Guarantee)
!python train.py --self-test

In [ ]:
# [STEP 4] Launch Full Multi-GPU Training on 2x Tesla T4
# - Automatically caches 10,400 seen photos (4-6x speedup: ~5.3 min/epoch)
# - Evaluates on 21 Unseen Classes (2,100 photos | 12,580 sketches) every epoch
# - Prints full diagnostic radar (12 academic metrics + 5 loss components)
!python train_kaggle_2xt4.py --epochs 15 --batch-size 128 --lora-rank 16 --triplet-margin 0.25

In [ ]:
# [STEP 5] Display Publication-Grade Training & Diagnostic Curves
import os
from IPython.display import Image, display
curve_path = 'outputs/training_curves.png'
if os.path.exists(curve_path):
    display(Image(curve_path))
else:
    print(f'Training curves image not found at {curve_path}')

In [ ]:
# [STEP 6] Compute Full In-Domain Academic Benchmark Table (Sketchy 21 Unseen Classes)
# Computes: mAP@all, mAP@200, P@100/200 (Std & Norm), Recall@1/5/10, OOD AUROC
!python scripts/benchmark_full_table.py

In [ ]:
# [STEP 7] Cross-Dataset Open-World Evaluation on TU-Berlin Extended
# Tests model trained on Sketchy directly against all 21,940 photos & 2,240 sketches of TU-Berlin
!python scripts/evaluate.py --checkpoint outputs/checkpoints/best_seco_lora_clip.pt --dataset tuberlin --split cross_dataset --batch-size 128 --viz-samples 5

In [ ]:
# [STEP 8] Package Trained Weights & Results for 1-Click Download
!zip -r lora_clip_cs331_outputs.zip outputs/
print("Packaging complete! You can download 'lora_clip_cs331_outputs.zip' from the Kaggle Output file explorer on the right panel.")